# NẠP DỮ LIỆU THÔ ĐÃ QUY ĐỔI NHÃN

In [1]:
import os
import re
import sys
import pandas as pd
import numpy as np

In [2]:
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
PROC_DIR = os.path.join(ROOT, 'data', 'processed')
if not os.environ.get('TEMP'):
    os.environ['TEMP'] = os.path.join(ROOT, '.tmp_run')
os.makedirs(os.environ['TEMP'], exist_ok=True)
os.environ['TMP'] = os.environ['TEMP']

In [3]:
#Đọc 3 tập dữ liệu từ bước 00
df_train = pd.read_csv(os.path.join(PROC_DIR, 'train_raw.csv'))
df_dev = pd.read_csv(os.path.join(PROC_DIR, 'dev_raw.csv'))
df_test = pd.read_csv(os.path.join(PROC_DIR, 'test_raw.csv'))

In [4]:
df_train.shape, df_dev.shape, df_test.shape

((7786, 21), (1112, 21), (2224, 21))

In [5]:
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 7786 entries, 0 to 7785
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   index              7786 non-null   int64
 1   sentence           7786 non-null   str  
 2   n_star             7786 non-null   int64
 3   date_time          7786 non-null   str  
 4   label              7786 non-null   str  
 5   sentiment          7786 non-null   int64
 6   aspect_group       7786 non-null   int64
 7   asp_0              7786 non-null   int64
 8   asp_1              7786 non-null   int64
 9   asp_2              7786 non-null   int64
 10  asp_3              7786 non-null   int64
 11  asp_4              7786 non-null   int64
 12  all_aspects        7786 non-null   str  
 13  absa_0             7786 non-null   int64
 14  absa_1             7786 non-null   int64
 15  absa_2             7786 non-null   int64
 16  absa_3             7786 non-null   int64
 17  absa_4             7786 n

In [6]:
df_train.head()

,index,sentence,n_star,date_time,label,sentiment,aspect_group,asp_0,asp_1,asp_2,...,asp_4,all_aspects,absa_0,absa_1,absa_2,absa_3,absa_4,has_conflict,conflict_with_neg,conflict_groups
0,0,Mới mua máy này Tại thegioididong thốt nốt cảm...,5,2 tuần trước,{CAMERA#Positive};{FEATURES#Positive};{BATTERY...,2,2,1,1,1,...,1,"0,1,2,3,4",3,3,3,3,3,False,False,NaN
1,1,Pin kém còn lại miễn chê mua 8/3/2019 tình trạ...,5,14/09/2019,{BATTERY#Negative};{GENERAL#Positive};{OTHERS};,1,2,0,0,1,...,1,"2,4",0,0,1,0,3,False,False,NaN
2,2,Sao lúc gọi điện thoại màn hình bị chấm nhỏ nh...,3,17/08/2020,{FEATURES#Negative};,0,0,1,0,0,...,0,0,1,0,0,0,0,False,False,NaN
3,3,"Mọi người cập nhật phần mềm lại , nó sẽ bớt tố...",3,29/02/2020,{FEATURES#Negative};{BATTERY#Neutral};{GENERAL...,1,0,1,0,1,...,1,"0,2,4",1,0,2,0,2,False,False,NaN
4,4,"Mới mua Sài được 1 tháng thấy pin rất trâu, Sà...",5,4/6/2020,{BATTERY#Positive};{PERFORMANCE#Positive};{SER...,2,3,1,0,1,...,0,"0,2,3",3,0,3,1,0,False,False,NaN


## LÀM SẠCH VĂN BẢN, ẨN DANH PII VÀ TÁCH TỪ PYVI

In [7]:
#Nạp hàm chuẩn hóa giữ hoa/thường, lọc câu rỗng, tách từ pyvi và hàm lọc near-dup từ src
from src.labels import ABSA_COLS, _strip_punct_words, dedup_remove_train_neardup
from src.text_cleaner import clean_text, check_no_content, segment_words


In [8]:
#Thử nghiệm trên 1 câu mẫu chứa PII, tiếng cười, từ tiếng Anh lặp chữ và từ có 'ưu'
tempt = 'TGDĐ đt đẹppp quáaaa, lưu ý freee full app call kkk màu cam affordable ORD-2024-001 sđt 090.123.4567 tại thegioididong.com ❤️'
tempt

'TGDĐ đt đẹppp quáaaa, lưu ý freee full app call kkk màu cam affordable ORD-2024-001 sđt 090.123.4567 tại thegioididong.com ❤️'

In [9]:
#Làm sạch văn bản giữ hoa/thường cho PhoBERT
tempt_clean = clean_text(tempt, lowercase=False)
tempt_clean

'Thế Giới Di Động điện thoại đẹp quá, lưu ý free full app call cười màu cam affordable [ORDER_ID] sđt [PHONE] tại [URL] yêu_thích'

In [10]:
#Kiểm tra tách từ pyvi có bảo toàn nguyên dấu ngoặc [PHONE], [ORDER_ID], [URL]
segment_words(tempt_clean)

'Thế_Giới_Di_Động điện_thoại đẹp quá , lưu_ý free full app call cười màu cam affordable [ORDER_ID] sđt [PHONE] tại [URL] yêu_thích'

In [11]:
#Kiểm tra hàm lọc câu rỗng hoặc săn xu trên mẫu thử
check_no_content('cho 5 sao nhận xu'), check_no_content('Máy dùng tốt'), check_no_content(tempt_clean)

(True, False, False)

## ÁP DỤNG TIỀN XỬ LÝ CHO 3 TẬP DỮ LIỆU

In [12]:
#Áp dụng làm sạch (cased), gắn cờ câu vô nghĩa và tách từ pyvi
for df in [df_train, df_dev, df_test]:
    df['sentence_clean'] = df['sentence'].apply(lambda x: clean_text(x, lowercase=False))
    df['is_no_content'] = df['sentence_clean'].apply(check_no_content)
    df['sentence_seg'] = df['sentence_clean'].apply(segment_words)

In [13]:
df_train['is_no_content'].value_counts()

is_no_content
False    7786
Name: count, dtype: int64

In [14]:
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 7786 entries, 0 to 7785
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   index              7786 non-null   int64
 1   sentence           7786 non-null   str  
 2   n_star             7786 non-null   int64
 3   date_time          7786 non-null   str  
 4   label              7786 non-null   str  
 5   sentiment          7786 non-null   int64
 6   aspect_group       7786 non-null   int64
 7   asp_0              7786 non-null   int64
 8   asp_1              7786 non-null   int64
 9   asp_2              7786 non-null   int64
 10  asp_3              7786 non-null   int64
 11  asp_4              7786 non-null   int64
 12  all_aspects        7786 non-null   str  
 13  absa_0             7786 non-null   int64
 14  absa_1             7786 non-null   int64
 15  absa_2             7786 non-null   int64
 16  absa_3             7786 non-null   int64
 17  absa_4             7786 n

In [15]:
df_train[['sentence', 'sentence_clean', 'sentence_seg', 'is_no_content']].head()

,sentence,sentence_clean,sentence_seg,is_no_content
0,Mới mua máy này Tại thegioididong thốt nốt cảm...,Mới mua máy này Tại thegioididong thốt nốt cảm...,Mới mua máy này Tại thegioididong thốt_nốt cảm...,False
1,Pin kém còn lại miễn chê mua 8/3/2019 tình trạ...,Pin kém còn lại miễn chê mua 8/3/2019 tình trạ...,Pin kém còn lại miễn chê mua 8 / 3 / 2019 tình...,False
2,Sao lúc gọi điện thoại màn hình bị chấm nhỏ nh...,Sao lúc gọi điện thoại màn hình bị chấm nhỏ nh...,Sao lúc gọi điện_thoại màn_hình bị chấm nhỏ nh...,False
3,"Mọi người cập nhật phần mềm lại , nó sẽ bớt tố...","Mọi người cập nhật phần mềm lại , nó sẽ bớt tố...","Mọi người cập_nhật phần_mềm lại , nó sẽ bớt tố...",False
4,"Mới mua Sài được 1 tháng thấy pin rất trâu, Sà...","Mới mua Sài được 1 tháng thấy pin rất trâu, Sà...","Mới mua Sài được 1 tháng thấy pin rất trâu , S...",False


## CHỐNG TRÙNG LẶP NỘI BỘ VÀ CHỐNG RÒ RỈ GIỮA CÁC TẬP

In [16]:
#Dedup nội bộ train theo khóa chuẩn hóa (_norm_key: lowercase + bỏ dấu câu); xóa cả nhóm nếu mâu thuẫn nhãn (gồm absa_*), cộng dòng typo đã duyệt
n_tr_raw = len(df_train)
lbl_cols = ['sentiment', 'aspect_group', 'all_aspects'] + ABSA_COLS
df_train['_norm_key'] = df_train['sentence_clean'].apply(lambda s: ' '.join(_strip_punct_words(s)))
grp_tr = df_train.groupby('_norm_key')[lbl_cols].transform('nunique').max(axis=1)
df_train = df_train[grp_tr == 1].drop_duplicates(subset=['_norm_key'], keep='first').drop(columns=['_norm_key'])
df_train = df_train[~df_train['index'].isin({5537})].reset_index(drop=True)
print('Train internal conflicting/dup dropped:', n_tr_raw - len(df_train))


Train internal conflicting/dup dropped: 11


In [17]:
#Xử lý trùng lặp nội bộ tập dev bằng cùng khóa _norm_key với train
n_de_raw = len(df_dev)
df_dev['_norm_key'] = df_dev['sentence_clean'].apply(lambda s: ' '.join(_strip_punct_words(s)))
grp_n = df_dev.groupby('_norm_key')[lbl_cols].transform('nunique').max(axis=1)
df_dev = df_dev[grp_n == 1].drop_duplicates(subset=['_norm_key'], keep='first').drop(columns=['_norm_key']).reset_index(drop=True)
print('Dev internal conflicting/dup dropped:', n_de_raw - len(df_dev))


Dev internal conflicting/dup dropped:

 2


In [18]:
#Xóa mọi dòng trong train có sentence_clean trùng với dev hoặc test để chống leakage 0%
s_dev = set(df_dev['sentence_clean'].fillna(''))
s_test = set(df_test['sentence_clean'].fillna(''))
drop_dev = int(df_train['sentence_clean'].isin(s_dev).sum())
drop_test = int(df_train['sentence_clean'].isin(s_test).sum())
df_train = df_train[~df_train['sentence_clean'].isin(s_dev | s_test)].reset_index(drop=True)
print('Train overlap dropped -> dev:', drop_dev, '| test:', drop_test, '| train remaining:', len(df_train))

Train overlap dropped -> dev: 3 | test: 5 | train remaining: 7767


In [19]:
#Xác minh 0 trùng lặp chéo (train/dev/test) và 0 trùng lặp nội bộ
print('train&dev, train&test, dev&test:', len(set(df_train['sentence_clean']) & s_dev), len(set(df_train['sentence_clean']) & s_test), len(s_dev & s_test))
print('dup nội bộ train/dev:', df_train['sentence_clean'].duplicated().sum(), df_dev['sentence_clean'].duplicated().sum())
assert not (s_dev & s_test), 'dev và test trùng nhau'


train&dev, train&test, dev&test:

 0 0 0
dup nội bộ train/dev: 0 0


In [20]:
#Gate 2 bước lọc near-duplicate (cosine >= 0.90 trên char_wb 3-5gram đã bỏ dấu câu/chấm lặp, >= 5 từ)
APPLY_NEARDUP_DROP = True
NEARDUP_KEEP_IDS = {3205, 3357, 3617, 5358, 5876, 6587, 7775}
df_train, df_neardup, max_cos = dedup_remove_train_neardup(
    df_train, {'dev': df_dev, 'test': df_test}, threshold=0.90, min_words=5,
    keep_ids=NEARDUP_KEEP_IDS, apply=APPLY_NEARDUP_DROP,
)
print('max_cosine quantiles (50/90/95/99/max):', np.round(np.quantile(max_cos, [0.5, 0.9, 0.95, 0.99, 1.0]), 4))
for th in [0.80, 0.90, 0.95]:
    print(f'  train eligible >= {th:.2f}: {int((max_cos >= th).sum())}')
neardup_csv = os.path.join(PROC_DIR, 'train_neardup_dropped.csv')
if df_neardup.empty:
    df_neardup = pd.DataFrame(columns=['train_idx', 'row_pos', 'max_cosine', 'matched_split', 'train_text', 'matched_ref_text', 'approved'])
df_neardup.to_csv(neardup_csv, index=False, encoding='utf-8-sig')
print('Saved audit list:', neardup_csv, '| candidates:', len(df_neardup), '| apply:', APPLY_NEARDUP_DROP)
df_neardup


max_cosine quantiles (50/90/95/99/max): [0.3704 0.483  0.5231 0.6297 1.    ]
  train eligible >= 0.80: 21
  train eligible >= 0.90: 11
  train eligible >= 0.95: 5
Saved audit list: D:\E-commerce\data\processed\train_neardup_dropped.csv | candidates: 11 | apply: True


,train_idx,row_pos,max_cosine,matched_split,train_text,matched_ref_text,approved
0,542,541,0.9417,test,"Sản phẩm nhỏ gọn, vừa tay thiết kế màu sắc đẹp...","Sản phẩm nhỏ gọn, vừa tay thiết kế màu sắc đẹp...",True
1,2660,2656,0.9515,test,Pin xài được lâu mà sạc cũng nhanh đầy nữa. Mì...,Pin xài được lâu mà sạc cũng nhanh đầy nữa. Mì...,True
2,2932,2928,1.0000,test,"Máy tự khởi động lại, vân tay chạm nóng hổi, n...","Máy tự khởi động lại, vân tay chạm nóng hổi, n...",True
3,3013,3009,0.9523,test,"Mọi thứ Tạm ổn trong tầm giá , con chip 665 ch...","Mọi thứ Tạm ổn trong tầm giá , con chip 665 ch...",True
4,3205,3201,0.9379,test,"Sản phẩm tốt, pin trâu, camera đủ dùng, âm tha...","Sản phẩm tốt, pin trâu, âm thanh sau cập nhật ...",False
5,3357,3352,0.9204,test,"Sản phẩm thiết kế sang trọng, màu sắc đẹp , bề...","Sản phẩm thiết kế sang trọng, màu sắc đẹp , bề...",False
6,3617,3611,0.9496,test,"thiết kế đẹp màu sang trọng, pin khoẻ, trải ng...","Thiết kế đẹp màu sang trọng, pin khoẻ, trải ng...",False
7,5358,5346,0.9337,dev,Sản phẩm dùng cho nghe gọi ok. Loa lớn. Màu sắ...,Sản phẩm dùng cho nghe gọi ok. Loa lớn. Màu sắ...,False
8,5876,5860,0.9247,test,"Thiết kế đẹp, cấu hình tốt, pin xài onscreen h...","Thiết kế đẹp, cấu hình tốt trong tầm giá, pin ...",False
9,6587,6569,1.0000,test,"Máy mượt, chụp hình tốt, riêng cục pin lên 360...","Máy mượt, chụp hình tốt, riêng cục pin lên 360...",False


## KIỂM ĐỊNH TRƯỚC/SAU LÀM SẠCH (TOKEN DIFF AUDIT)

In [21]:
#So sánh số lỗi regex phổ biến trên toàn bộ 3 tập trước và sau khi qua src.text_cleaner
all_raw = pd.concat([df_train['sentence'], df_dev['sentence'], df_test['sentence']], ignore_index=True).fillna('')
all_clean = pd.concat([df_train['sentence_clean'], df_dev['sentence_clean'], df_test['sentence_clean']], ignore_index=True).fillna('')
all_seg = pd.concat([df_train['sentence_seg'], df_dev['sentence_seg'], df_test['sentence_seg']], ignore_index=True).fillna('')
print('luu/lu broken (lư ý|hữ) -> raw:', int(all_raw.str.contains(r'(?<!\w)(?:lư ý|hữ)(?!\w)', case=False).sum()), '| clean:', int(all_clean.str.contains(r'(?<!\w)(?:lư ý|hữ)(?!\w)', case=False).sum()))
print('full/app/call broken (ful|ap|cal) -> raw:', int(all_raw.str.contains(r'(?<!\w)(?:ful|ap|cal)(?!\w)', case=False).sum()), '| clean:', int(all_clean.str.contains(r'(?<!\w)(?:ful|ap|cal)(?!\w)', case=False).sum()))
print('tgdd/tgdđ unnormalized in clean:', int(all_clean.str.contains(r'(?<!\w)tgdđ?(?!\w)', case=False).sum()))

luu/lu broken (lư ý|hữ) -> raw: 0 | clean: 0


full/app/call broken (ful|ap|cal) -> raw: 5 | clean: 5


tgdd/tgdđ unnormalized in clean: 0


In [22]:
#Kiểm tra tiếp kkk, affordable, ngoặc [PHONE] sau pyvi và bộ đếm is_no_content
print('kkk remaining in clean:', int(all_clean.str.contains(r'(?<!\w)k{3,}(?!\w)', case=False).sum()))
print('affordable broken (af [ORDER_ID]):', int(all_clean.str.contains(r'af\s*\[ORDER_ID\]', case=False).sum()))
print('broken PII brackets in seg ([ PHONE ]):', int(all_seg.str.contains(r'\[\s+(?:URL|EMAIL|ORDER_ID|PHONE|PERSON)|(?:URL|EMAIL|ORDER_ID|PHONE|PERSON)\s+\]').sum()))
print('is_no_content counts (train/dev/test):', int(df_train['is_no_content'].sum()), int(df_dev['is_no_content'].sum()), int(df_test['is_no_content'].sum()))

kkk remaining in clean: 0
affordable broken (af [ORDER_ID]): 0
broken PII brackets in seg ([ PHONE ]): 0
is_no_content counts (train/dev/test): 0 0 0


## LƯU CÁC TẬP DỮ LIỆU ĐÃ LÀM SẠCH

In [23]:
import hashlib
# Nếu dữ liệu nguồn đổi thì NEARDUP_KEEP_IDS / {5537} gõ cứng không còn đúng -> dừng, duyệt lại thủ công
_fp = hashlib.sha256(','.join(map(str, sorted(df_train['index']))).encode()).hexdigest()[:16]
assert len(df_train) == 7762 and _fp == 'ad1b0833a88851e2', f'Train sau dedup thay đổi (n={len(df_train)}, fp={_fp}): duyệt lại các ID gõ cứng'
#Chỉ xuất *_clean.csv khi đã chốt APPLY_NEARDUP_DROP=True (bước 1 audit giữ nguyên *_clean.csv cũ)
if APPLY_NEARDUP_DROP:
    df_train.to_csv(os.path.join(PROC_DIR, 'train_clean.csv'), index=False, encoding='utf-8-sig')
    df_dev.to_csv(os.path.join(PROC_DIR, 'dev_clean.csv'), index=False, encoding='utf-8-sig')
    df_test.to_csv(os.path.join(PROC_DIR, 'test_clean.csv'), index=False, encoding='utf-8-sig')
    print('Exported clean CSVs:', df_train.shape, df_dev.shape, df_test.shape)
else:
    print('AUDIT MODE (APPLY_NEARDUP_DROP=False): cho duyet train_neardup_dropped.csv truoc khi ghi train_clean.csv')


Exported clean CSVs: (7762, 24) (1110, 24) (2224, 24)


In [24]:
os.listdir(PROC_DIR)

['actionable_insights.csv',
 'dev_clean.csv',
 'dev_pred_phobert.csv',
 'dev_raw.csv',
 'sample_excel_predicted.xlsx',
 'test_clean.csv',
 'test_pred_baseline.csv',
 'test_pred_phobert.csv',
 'test_raw.csv',
 'train_clean.csv',
 'train_internal_neardup_audit.csv',
 'train_neardup_dropped.csv',
 'train_raw.csv',
 'visfd_merged_raw.csv']